### Low rank analysis - State vector
Import libraries

In [ ]:
import sys

libs = {
  "./../modules",
  "/u/zanardi3/Codes/ML/PyCOMET/pycomet"
}
for lib in libs:
  sys.path.append(lib)

In [ ]:
import os
import json
import numpy as np
import dill as pickle
import dask.array as da

In [ ]:
import reading as read_mod
import plotting as plt_mod

from pycomet.networks.elements import LinAutoencoder

Define inputs

In [ ]:
# identifier = "vel1"
# # Path to saved solution
# path_to_sol = f"./../output/data/{identifier}.p"
# # Path to saving folder
# path_to_save = f"./../output/lowrank_state/{identifier}/"
# # Maximum number of reduced dimensions
# rdim_max = 10
# # Starting iteration considered
# iter0 = 5
# # Lower bounds
# bounds_low = {
#   "Th": 5e3,
#   "Te": 5e3,
#   "dT": -1e2
# }

In [ ]:
identifier = "vel1_i100"
# Path to saved solution
path_to_sol = f"./../output/data/{identifier}.p"
# Path to saving folder
path_to_save = f"./../output/lowrank_state/{identifier}/"
# Maximum number of reduced dimensions
rdim_max = 10
# Starting iteration considered
iter0 = 0
# Lower bounds
bounds_low = {
  "Th": 2.5e3,
  "Te": 2.5e2,
  "dT": 0.0
}

In [ ]:
os.makedirs(path_to_save, exist_ok=True)

Read saved solution

In [ ]:
dset = read_mod.load_dataset(path_to_sol)

Clean data

> Remove initial states

In [ ]:
for (k, v) in dset.items():
  if (k not in ("x_grid",)):
    dset[k] = v[iter0:]

> Extract post-shock states

In [ ]:
i_ps = dset["p"] > dset["p"][0,0]
for (k, v) in dset.items():
  if (k not in ("x_grid", "t_grid")):
    dset[k] = v[i_ps]

> Remove unphysical solutions

In [ ]:
dset["dT"] = dset["Th"] - dset["Te"]
i_dT = dset["dT"] > 0.0
for (k, v) in dset.items():
  if (k not in ("x_grid", "t_grid")):
    dset[k] = v[i_dT]

Compress mass fractions (i.e., distribution)
> Transform

In [ ]:
w = dset["w"]
wln = np.log(w)
wln_m = np.mean(wln, axis=0, keepdims=True)
wln_s = np.std(wln, axis=0, keepdims=True)
wln_n = (wln-wln_m)/wln_s

> Perform SVD

In [ ]:
yd = da.from_array(wln_n.T, chunks="auto")
u, s, vh = da.linalg.svd_compressed(yd, k=2*rdim_max)
s = s.compute()
u = u.compute()

> Singular value decay

In [ ]:
# Normalized cumulative sum
cs = 1.0 / np.sum(s**2)
cs *= np.cumsum(s**2)
# Number of principal components
eps = 1e-5
rdim = np.where(cs > 1-eps)[0][0] + 1
np.savetxt(path_to_save + "/rdim.txt", np.array(rdim).reshape(1))
print("Number of reduced dimensions:", rdim)

In [ ]:
plt_mod.cumenergy(
  cs[:rdim_max],
  figname=path_to_save + "/cum_en.pdf",
  save=True,
  show=False
)

> Check reconstruction error

In [ ]:
rom = LinAutoencoder(
  phi=u[:,:rdim],
  psi=u[:,:rdim],
  xref=wln_m,
  xscale=wln_s,
  use_log=True
)

In [ ]:
wr = rom(w, method=3).numpy()
err = 100*np.abs(wr-w)/(w+1e-15)
print("Max pointwise error: %.2f" % err.max(), "%")

In [ ]:
with open(path_to_save+"/rom.p", "wb") as file:
  pickle.dump(rom.get_config(), file)

Compute bounds

> From data

In [ ]:
bounds = {k: [dset[k].min(), dset[k].max()] for k in ("rho", "Th", "Te", "dT")}
# Latent space
z = rom(w, method=1).numpy()
for i in range(rdim):
  ki = f"z{i+1}"
  zi = z[:,i]
  dset[ki] = zi
  bounds[ki] = [zi.min(), zi.max()]
# Save bounds
with open(path_to_save+"/bounds_data.json", "w") as file:
  json.dump(bounds, file, indent=2)

> From inputs

In [ ]:
# Set bounds
for (k, v) in bounds_low.items():
  bounds[k][0] = v
# Save bounds
with open(path_to_save+"/bounds.json", "w") as file:
  json.dump(bounds, file, indent=2)

Save dataset

In [ ]:
with open(path_to_save+"/dset.p", "wb") as file:
  pickle.dump(dset, file)